# Z1: how sensitive is Fair Bin to z?

As `robust_zsweep.py`: Fair Bin with z swept over 0 … 12 and the Bonferroni z, at k = 5, 10 and 14. Each seed's
error counts are computed once from its training rows and shared by every z and k. The bins are scored with a
random forest on the test rows (`metrics.evaluate`).

- `regret_pct`: how far a dataset's MRE at a given z is above its best z, in % (mean over seeds first).
- Figures: `figures/Z1_regret_vs_z.{png,pdf}` and `figures/Z1_smallest_bin_vs_z.{png,pdf}`.

**Tally.** The published Z1 used out-of-fold **random-forest** counts (`robust.oof_tally`), from before Fair Bin
switched to the stump. `TALLY = "stump"` (the default) is Fair Bin as it is now; `TALLY = "rf"` reproduces the
published figure.

**Run time:** several hours on all 15 datasets (13 z × 3 k × 9 seeds; KDD dominates). Set `DATASETS` to a subset
to try it.

In [ ]:
import random
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import datasets, fairbin, figures, metrics

OUT = ROOT / "figures"
OUT.mkdir(exist_ok=True)
SEEDS = [92, 50, 61, 19, 11, 8, 2, 51, 70]  # the paper's nine eval seeds
DATASETS = datasets.group_names("uci")      # the 15 UCI datasets
KS = (5, 10, 14)
Z_VALUES = (0, 0.5, 1, 1.5, 1.96, 2.5, 3, 4, 5, 6, 8, 12, "bonf")
TALLY = "stump"                             # "rf": out-of-fold random forest counts, as the published Z1

## Error counts
`fairbin.tally` for the stump; for `TALLY = "rf"`, `robust.oof_predictions` (copied): the same folds with a random forest.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict


def rf_oof_predictions(train, y_name, seed, folds=fairbin.FOLDS):
    X = train.drop(columns=[y_name])
    y = train[y_name].to_numpy()
    splitter = StratifiedKFold if pd.Series(y).value_counts().min() >= folds else KFold
    clf = RandomForestClassifier(random_state=seed, n_jobs=-1)
    return cross_val_predict(clf, X, y, cv=splitter(n_splits=folds, shuffle=True, random_state=seed))


def counts(train, spec, seed):
    if TALLY == "stump":
        return fairbin.tally(train, spec.target, spec.bin_col, seed)
    pred = rf_oof_predictions(train, spec.target, seed)
    return fairbin.error_counts(train[spec.bin_col], train[spec.target], pred)


def score(data, spec, n_classes, seed, cuts, cache):
    """metrics.evaluate (rf) on cuts, each distinct cut vector once per (dataset, seed)."""
    key = tuple(cuts)
    if key not in cache:
        binned = fairbin.apply_bins(data[spec.bin_col], cuts).to_numpy(float)
        bin_idx, layout = metrics.bin_layout(binned, cuts)
        ev = metrics.evaluate(data, spec.bin_col, spec.target, binned, bin_idx, len(cuts) - 1, "rf", seed, n_classes == 2)
        cache[key] = {"mre": ev["mre"], "accuracy": ev["accuracy"], "smallest_bin_share": layout["smallest_bin_share"]}
    return cache[key]

## Run

In [ ]:
out = []
start = time.perf_counter()
for ds in DATASETS:
    data, spec, n_classes = datasets.load(ds)
    for seed in SEEDS:
        train = data.loc[metrics.split(data, spec.target, seed)[0].index]
        values, errors, rows = counts(train, spec, seed)
        z_bonf = fairbin.bonferroni_z(len(values))
        cache = {}
        for k in KS:
            for zv in Z_VALUES:
                z = z_bonf if zv == "bonf" else float(zv)
                cuts = fairbin.choose_cuts(values, errors, rows, k, z)[1]
                out.append({"dataset": ds, "k": k, "z_label": str(zv), "z": z, "eval_seed": seed, "bin_cuts": cuts,
                            **score(data, spec, n_classes, seed, cuts, cache)})
    print(f"[{(time.perf_counter() - start) / 60:5.1f}m] {ds} done", flush=True)
runs = pd.DataFrame(out)

## Summary (as `robust_zsweep.summarize`)

In [ ]:
s = runs.groupby(["dataset", "k", "z_label"]).agg(
    z=("z", "mean"), mre=("mre", "mean"), accuracy=("accuracy", "mean"),
    smallest_bin_share=("smallest_bin_share", "median"), n_seeds=("mre", "size")).reset_index()
s["regret_pct"] = 100 * (s.mre / s.groupby(["dataset", "k"]).mre.transform("min") - 1)
labels = [str(z) for z in Z_VALUES]
s.pivot_table(index="z_label", columns="k", values="regret_pct", aggfunc="median").reindex(labels).round(1)

## Figures

In [ ]:
figures.z_style()
figures.z1(s, OUT)
for name in ("Z1_regret_vs_z", "Z1_smallest_bin_vs_z"):
    display(Image(str(OUT / f"{name}.png")))